In [ ]:
# DAY 2 MORNING X-RAY: See how an AI Brain actually reads human words!
from transformers import AutoTokenizer

# 1. Hire the Translator (Tokenizer) for Qwen
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-1.5B-Instruct")

# 2. Give it a messy human sentence
sentence = "Invoice #9042: Customer Diman paid $1,250.50 for AI Architecture!"

# 3. Chop the sentence into Lego bricks (Tokens)
lego_bricks = tokenizer.tokenize(sentence)
print("🧱 STEP 1 - LEGO BRICKS (Tokens):")
print(lego_bricks)

# 4. Turn those Lego bricks into Secret Barcode Numbers (input_ids)
barcode_numbers = tokenizer.encode(sentence)
print("\n🔢 STEP 2 - BARCODE NUMBERS (input_ids):")
print(barcode_numbers)

# 5. Translate a single barcode number back into English to prove how it works!
print("\n🔍 STEP 3 - DECODING BARCODE ONE BY ONE:")
for num in barcode_numbers:
    print(f"Barcode {num:<8} ---> '{tokenizer.decode([num])}'")

In [ ]:
# CELL 2: See the hidden "Walkie-Talkie" tags (Chat Templates) inside AI Agents!
messages = [
    {"role": "system", "content": "You are a strict bank accountant."},
    {"role": "user", "content": "How much did Diman pay?"}
]

# Turn the conversation into the secret script the AI Brain actually sees
secret_prompt = tokenizer.apply_chat_template(
    messages, 
    tokenize=False, 
    add_generation_prompt=True
)

print("🕵️‍♂️ WHAT THE AI BRAIN ACTUALLY SEES BEHIND THE SCENES:")
print(secret_prompt)

In [ ]:
# CELL 3: See how the Tokenizer packs uneven sentences into a GPU Egg Carton (Tensor)!
batch_sentences = [
    "Diman paid.",                                  # Short sentence
    "Customer Diman paid $1,250.50 for AI today."   # Long sentence
]

# Pack both into a PyTorch Tensor ("pt") and fill empty holes with plastic eggs (padding=True)
egg_carton = tokenizer(batch_sentences, padding=True, return_tensors="pt")

print("🥚 1. THE EGG CARTON (input_ids - Notice the repeated filler numbers in Row 1!):")
print(egg_carton["input_ids"])

print("\n🕶️ 2. THE BLINDFOLD (attention_mask - 1 = Real Word, 0 = Ignore Filler Egg!):")
print(egg_carton["attention_mask"])

In [ ]:
# CELL 4: X-Ray vision into how @tool turns a Python function into an AI Instruction Card!
!pip install -q smolagents
from smolagents import tool

@tool
def lookup_customer_limit(customer_name: str) -> int:
    """
    Looks up the maximum allowed spending limit (in USD) for a registered bank customer.
    Always call this tool FIRST before approving any invoice.

    Args:
        customer_name: The first name of the customer (for example, 'Diman' or 'Alice').
    """
    database = {"Diman": 5000, "Alice": 1000, "Bob": 250}
    return database.get(customer_name, 0)

print("🪪 1. TOOL NAME:", lookup_customer_limit.name)
print("\n📋 2. WHAT THE AI SEES FOR INPUTS:", lookup_customer_limit.inputs)
print("\n🎯 3. OUTPUT TYPE:", lookup_customer_limit.output_type)
print("\n🤖 4. FULL INSTRUCTION CARD SENT TO THE AI BRAIN:")
print(lookup_customer_limit.description)

In [ ]:
# CELL 5: Build & Test our 3-Tool Fraud Detective Belt on Free CPU!
from smolagents import tool

# TOOL 1: The Filing Cabinet (Database Lookup)
@tool
def lookup_customer_limit(customer_name: str) -> int:
    """
    Looks up the maximum allowed spending limit in USD for a customer.
    Call this tool FIRST to find out how much the customer is allowed to spend.

    Args:
        customer_name: First name of the customer (e.g., 'Diman', 'Alice', 'Bob').
    """
    limits_db = {"Diman": 5000, "Alice": 1000, "Bob": 250}
    return limits_db.get(customer_name, 0)

# TOOL 2: The Currency Calculator (LKR to USD)
@tool
def convert_lkr_to_usd(amount_lkr: float) -> float:
    """
    Converts Sri Lankan Rupees (LKR) into US Dollars (USD) using a fixed rate of 300 LKR = 1 USD.
    Call this tool SECOND whenever an invoice amount is given in LKR.

    Args:
        amount_lkr: The invoice bill amount in Sri Lankan Rupees (LKR).
    """
    return round(amount_lkr / 300.0, 2)

# TOOL 3: The Alarm Button (Fraud Checker)
@tool
def evaluate_invoice_risk(amount_usd: float, limit_usd: int) -> str:
    """
    Compares the invoice amount in USD against the customer's limit in USD.
    Call this tool LAST to get the final APPROVED or FRAUD_ALERT decision.

    Args:
        amount_usd: The invoice total converted into USD.
        limit_usd: The customer's maximum allowed spending limit in USD.
    """
    if amount_usd <= limit_usd:
        return f"APPROVED: ${amount_usd} is within the ${limit_usd} limit."
    else:
        over_by = round(amount_usd - limit_usd, 2)
        return f"FRAUD_ALERT: ${amount_usd} exceeds the ${limit_usd} limit by ${over_by}!"

# --- PRO ARCHITECT TEST: Test all 3 tools in pure Python before waking up the AI! ---
print("🧪 Manual Tool Test for Alice (Invoice: 450,000 LKR):")
alice_limit = lookup_customer_limit("Alice")
usd_bill = convert_lkr_to_usd(450000.0)
verdict = evaluate_invoice_risk(usd_bill, alice_limit)

print(f"1. Alice's Limit: ${alice_limit}")
print(f"2. 450,000 LKR in USD: ${usd_bill}")
print(f"3. Final Verdict: {verdict}")

In [1]:
# CELL 6: Autonomous 3-Tool Detective Agent on T4 GPU
!pip install -q smolagents transformers accelerate torch

import torch
from smolagents import CodeAgent, TransformersModel, tool

# 1. DEFINE OUR 3 DETECTIVE TOOLS
@tool
def lookup_customer_limit(customer_name: str) -> int:
    """
    Looks up the maximum allowed spending limit in USD for a customer.
    Call this tool FIRST to find out how much the customer is allowed to spend.

    Args:
        customer_name: First name of the customer (e.g., 'Diman', 'Alice', 'Bob').
    """
    limits_db = {"Diman": 5000, "Alice": 1000, "Bob": 250}
    return limits_db.get(customer_name, 0)

@tool
def convert_lkr_to_usd(amount_lkr: float) -> float:
    """
    Converts Sri Lankan Rupees (LKR) into US Dollars (USD) using a fixed rate of 300 LKR = 1 USD.
    Call this tool SECOND whenever an invoice amount is given in LKR.

    Args:
        amount_lkr: The invoice bill amount in Sri Lankan Rupees (LKR).
    """
    return round(amount_lkr / 300.0, 2)

@tool
def evaluate_invoice_risk(amount_usd: float, limit_usd: int) -> str:
    """
    Compares the invoice amount in USD against the customer's limit in USD.
    Call this tool LAST to get the final APPROVED or FRAUD_ALERT decision.

    Args:
        amount_usd: The invoice total converted into USD.
        limit_usd: The customer's maximum allowed spending limit in USD.
    """
    if amount_usd <= limit_usd:
        return f"APPROVED: ${amount_usd} is within the ${limit_usd} limit."
    else:
        over_by = round(amount_usd - limit_usd, 2)
        return f"FRAUD_ALERT: ${amount_usd} exceeds the ${limit_usd} limit by ${over_by}!"

# 2. LOAD THE QWEN-3B BRAIN ONTO THE T4 GPU
print("🧠 Loading Qwen2.5-3B-Instruct onto T4 GPU...")
model = TransformersModel(
    model_id="Qwen/Qwen2.5-3B-Instruct",
    device_map="auto",
    torch_dtype=torch.float16,
    max_new_tokens=512,
)

# 3. INITIALIZE THE DETECTIVE AGENT
detective_agent = CodeAgent(
    tools=[lookup_customer_limit, convert_lkr_to_usd, evaluate_invoice_risk],
    model=model,
    max_steps=5
)

# 4. RUN TEST CASES
case_1 = (
    "Investigate this invoice: Customer Alice just submitted a bill for 450,000 LKR. "
    "Use your tools to check her USD limit, convert the LKR bill to USD, and give the final risk verdict."
)
print("\n🕵️‍♂️ --- EXECUTING CASE 1: ALICE ---")
verdict_1 = detective_agent.run(case_1)
print(f"\n✅ FINAL VERDICT (ALICE): {verdict_1}")

case_2 = (
    "Investigate this invoice: Customer Diman submitted a bill for 600,000 LKR. "
    "Use your tools to check his USD limit, convert the LKR bill to USD, and give the final risk verdict."
)
print("\n🕵️‍♂️ --- EXECUTING CASE 2: DIMAN ---")
verdict_2 = detective_agent.run(case_2)
print(f"\n✅ FINAL VERDICT (DIMAN): {verdict_2}")

🧠 Loading Qwen2.5-3B-Instruct onto T4 GPU...


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/35.6k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

... (27 outputs left out)

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  final_answer(risk_decision)                                                                                      
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: APPROVED: $2000.0 is within the $5000 limit.

[Step 2: Duration 7.34 seconds| Input tokens: 5,004 | Output tokens: 310]


✅ FINAL VERDICT (DIMAN): APPROVED: $2000.0 is within the $5000 limit.
